# 102 — TCGA RNA-seq Cohort Freeze

## Objective

This notebook freezes the exact TCGA primary tumor RNA-seq cohort used as the tumor discovery input for the project.

The cohort was constructed manually through the GDC Data Portal using file-level filters designed to retrieve open-access TCGA RNA-seq gene expression quantification files generated with the STAR Counts workflow.

Because GDC queries are dynamic and may change over time as metadata are updated, this notebook does not rely only on a textual description of the filters. Instead, it generates version-controlled cohort definition files that preserve the exact file-level cohort obtained at download time.

## GDC Cohort Definition

The cohort was obtained from the GDC Data Portal using the following filters:

* Program: TCGA
* Data Category: Transcriptome Profiling
* Data Type: Gene Expression Quantification
* Experimental Strategy: RNA-Seq
* Workflow Type: STAR - Counts
* Access: Open
* Tumor Descriptor: Primary

The resulting file-level cohort contained:

* 10,308 RNA-seq STAR Counts files
* 10,328 associated TCGA cases reported by the GDC portal
* 43.67 GB total reported file size

The downloaded GDC files used as raw provenance inputs are:

* `gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt`
* `gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv`
* `gdc_metadata_tcga_primary_tumor_rnaseq_star_counts.json`

## Outputs

This notebook generates lightweight, version-controlled cohort freeze files under:

`config/manifests/`

The intended outputs are:

* `gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt`
* `gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv`
* `gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv`
* `gdc_cohort_freeze_tcga_primary_tumor_rnaseq_star_counts.json`

These files define the exact TCGA RNA-seq cohort used by the project and allow downstream users to reproduce the same file-level download with the GDC Data Transfer Tool, provided that the referenced GDC file UUIDs remain available.

## Scientific Scope

This cohort is used for tumor-level discovery of recurrent transcriptomic programs. It is not used to infer clinical resistance, causal mechanisms, or therapeutic response. Downstream analyses should treat any associations derived from this cohort as computational and hypothesis-generating.

---


In [1]:
# =============================================================================
# 102 — TCGA RNA-seq Cohort Freeze
# Imports and path configuration
# =============================================================================

import json
import shutil
import sys
from pathlib import Path

import pandas as pd

In [2]:
# =============================================================================
# Project utilities
# =============================================================================

# Allow imports from src/ when running this notebook from the notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "pancancer-epigenetics":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate project root directory.")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import Paths

In [3]:
# =============================================================================
# Raw GDC provenance inputs
# =============================================================================

RAW_TCGA_DIR = Paths.tcga

RAW_MANIFEST_PATH = RAW_TCGA_DIR / "gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt"
RAW_SAMPLE_SHEET_PATH = RAW_TCGA_DIR / "gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv"
RAW_METADATA_PATH = RAW_TCGA_DIR / "gdc_metadata_tcga_primary_tumor_rnaseq_star_counts.json"

In [4]:
# =============================================================================
# Version-controlled cohort freeze outputs
# =============================================================================

CONFIG_MANIFEST_DIR = Paths.config / "manifests"
CONFIG_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

OUT_MANIFEST_PATH = CONFIG_MANIFEST_DIR / "gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt"
OUT_SAMPLE_SHEET_PATH = CONFIG_MANIFEST_DIR / "gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv"
OUT_FILE_INDEX_PATH = CONFIG_MANIFEST_DIR / "gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv"
OUT_COHORT_FREEZE_PATH = CONFIG_MANIFEST_DIR / "gdc_cohort_freeze_tcga_primary_tumor_rnaseq_star_counts.json"

In [5]:
# =============================================================================
# Display configured paths
# =============================================================================

print("Project root:")
print(Paths.root)

print("\nRaw TCGA input files:")
print(RAW_MANIFEST_PATH)
print(RAW_SAMPLE_SHEET_PATH)
print(RAW_METADATA_PATH)

print("\nVersion-controlled cohort freeze outputs:")
print(OUT_MANIFEST_PATH)
print(OUT_SAMPLE_SHEET_PATH)
print(OUT_FILE_INDEX_PATH)
print(OUT_COHORT_FREEZE_PATH)

Project root:
C:\proyectos\pancancer-epigenetics

Raw TCGA input files:
C:\proyectos\pancancer-epigenetics\data\raw\tcga\gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt
C:\proyectos\pancancer-epigenetics\data\raw\tcga\gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv
C:\proyectos\pancancer-epigenetics\data\raw\tcga\gdc_metadata_tcga_primary_tumor_rnaseq_star_counts.json

Version-controlled cohort freeze outputs:
C:\proyectos\pancancer-epigenetics\config\manifests\gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt
C:\proyectos\pancancer-epigenetics\config\manifests\gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv
C:\proyectos\pancancer-epigenetics\config\manifests\gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv
C:\proyectos\pancancer-epigenetics\config\manifests\gdc_cohort_freeze_tcga_primary_tumor_rnaseq_star_counts.json


In [6]:
# =============================================================================
# Validate raw input file availability
# =============================================================================

raw_input_paths = {
    "GDC manifest": RAW_MANIFEST_PATH,
    "GDC sample sheet": RAW_SAMPLE_SHEET_PATH,
    "GDC metadata JSON": RAW_METADATA_PATH,
}

missing_files = []

for label, path in raw_input_paths.items():
    exists = path.exists()
    size_mb = path.stat().st_size / 1024**2 if exists else None

    print(f"{label}:")
    print(f"  path:   {path}")
    print(f"  exists: {exists}")
    if exists:
        print(f"  size:   {size_mb:.2f} MB")
    else:
        missing_files.append(path)
    print()

if missing_files:
    raise FileNotFoundError(
        "Missing required raw TCGA provenance input files:\n"
        + "\n".join(str(path) for path in missing_files)
    )

GDC manifest:
  path:   C:\proyectos\pancancer-epigenetics\data\raw\tcga\gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt
  exists: False

GDC sample sheet:
  path:   C:\proyectos\pancancer-epigenetics\data\raw\tcga\gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv
  exists: True
  size:   2.36 MB

GDC metadata JSON:
  path:   C:\proyectos\pancancer-epigenetics\data\raw\tcga\gdc_metadata_tcga_primary_tumor_rnaseq_star_counts.json
  exists: False



FileNotFoundError: Missing required raw TCGA provenance input files:
C:\proyectos\pancancer-epigenetics\data\raw\tcga\gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt
C:\proyectos\pancancer-epigenetics\data\raw\tcga\gdc_metadata_tcga_primary_tumor_rnaseq_star_counts.json